# CINESCOPE — IMDb Review Sentiment Classification

**SI Lab extension** · Original IMDb 50K reviews · TF-IDF + Linear SVM / Logistic Regression · Google Colab

Predict whether a written review is **positive or negative**. This is separate from the original **0–10 movie rating regression** task. We use every valid labeled review, train/validation/test splits, and report the actual untouched test result. No score is hardcoded.

## 1. Setup and download

In [ ]:
%pip -q install kagglehub scikit-learn==1.6.1 joblib matplotlib seaborn

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import kagglehub
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)

RANDOM_STATE = 42
path = Path(kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"))
files_found = list(path.rglob("IMDB Dataset.csv"))
assert files_found, f"IMDB Dataset.csv not found in {path}"
df = pd.read_csv(files_found[0])
print("Dataset shape:", df.shape)
display(df.head(3))

## 2. Inspect the original reviews

In [ ]:
print(df.info())
print("Missing values:\n", df.isna().sum())
print("Sentiment counts:\n", df["sentiment"].value_counts())
print("Duplicate rows:", df.duplicated().sum())
assert df["review"].notna().all() and df["sentiment"].notna().all(), "Dataset contains missing reviews or labels"
assert set(df["sentiment"].unique()) == {"positive", "negative"}
assert df["review"].astype(str).str.strip().ne("").all()

df["review_length"] = df["review"].str.split().str.len()
print(df["review_length"].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="sentiment", order=["negative", "positive"], ax=axes[0], color="#a78bfa")
axes[0].set_title("Sentiment distribution")
sns.histplot(data=df, x="review_length", hue="sentiment", bins=45, ax=axes[1], element="step")
axes[1].set_title("Review length distribution")
axes[1].set_xlim(0, 1200)
plt.tight_layout()
plt.show()

## 3. Split before fitting TF-IDF

Approximately 80% train pool / 20% untouched test. Duplicate review texts stay in the same split, so identical reviews cannot leak across train and test. The vocabulary is learned **only** from training text.

In [ ]:
X = df["review"].astype(str)
y = df["sentiment"].map({"negative": 0, "positive": 1}).astype(int)

train_idx, test_idx = next(GroupShuffleSplit(test_size=0.20, random_state=RANDOM_STATE).split(X, y, groups=X))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
fit_idx, val_idx = next(GroupShuffleSplit(test_size=0.20, random_state=RANDOM_STATE).split(X_train, y_train, groups=X_train))
X_fit, X_val = X_train.iloc[fit_idx], X_train.iloc[val_idx]
y_fit, y_val = y_train.iloc[fit_idx], y_train.iloc[val_idx]

assert set(X_train).isdisjoint(set(X_test))
assert set(X_fit).isdisjoint(set(X_val))
print("Train pool:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))
print("Test label balance:", y_test.value_counts(normalize=True).to_dict())
print("Majority-class baseline:", round(y_test.value_counts(normalize=True).max(), 4))

## 4. TF-IDF and validation comparison

In [ ]:
vectorizer_args = {
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 150000,
    "sublinear_tf": True,
    "strip_accents": "unicode",
    "dtype": np.float32,
}

vectorizer = TfidfVectorizer(**vectorizer_args)
X_fit_vec = vectorizer.fit_transform(X_fit)
X_val_vec = vectorizer.transform(X_val)
print("TF-IDF shape:", X_fit_vec.shape)

candidates = {
    "Linear SVM (C=0.5)": LinearSVC(C=0.5, random_state=RANDOM_STATE, max_iter=3000),
    "Linear SVM (C=1.5)": LinearSVC(C=1.5, random_state=RANDOM_STATE, max_iter=3000),
    "Logistic Regression (C=4)": LogisticRegression(C=4, solver="liblinear", max_iter=1000, random_state=RANDOM_STATE),
}

results = []
for name, classifier in candidates.items():
    classifier.fit(X_fit_vec, y_fit)
    predictions = classifier.predict(X_val_vec)
    results.append({
        "Model": name,
        "Validation Accuracy": accuracy_score(y_val, predictions),
        "Validation F1": f1_score(y_val, predictions),
    })

comparison = pd.DataFrame(results).sort_values(
    ["Validation Accuracy", "Validation F1"], ascending=False
).reset_index(drop=True)
display(comparison.style.format({"Validation Accuracy": "{:.4f}", "Validation F1": "{:.4f}"}))
best_name = comparison.loc[0, "Model"]
print("Selected on validation data:", best_name)

## 5. Final model and held-out test evaluation

Refit the winning algorithm on the full original 80% training split. Test predictions are made **once** after model selection.

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(**vectorizer_args)),
    ("classifier", clone(candidates[best_name])),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
decision_scores = model.decision_function(X_test)

metrics = {
    "dataset": "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
    "prediction_task": "IMDb review sentiment (positive vs negative)",
    "selected_model": best_name,
    "train_size": int(len(X_train)),
    "validation_size": int(len(X_val)),
    "test_size": int(len(X_test)),
    "random_state": RANDOM_STATE,
    "accuracy": float(accuracy_score(y_test, predictions)),
    "precision": float(precision_score(y_test, predictions)),
    "recall": float(recall_score(y_test, predictions)),
    "f1": float(f1_score(y_test, predictions)),
    "roc_auc": float(roc_auc_score(y_test, decision_scores)),
    "majority_baseline_accuracy": float(y_test.value_counts(normalize=True).max()),
    "features": "TF-IDF unigram + bigram terms from full review text",
}
print(pd.Series(metrics))
print("\nClassification report:\n", classification_report(y_test, predictions, target_names=["negative", "positive"]))

In [ ]:
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
fpr, tpr, _ = roc_curve(y_test, decision_scores)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=axes[0],
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Test-set confusion matrix")
axes[1].plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("Test-set ROC curve")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Interpret the words driving predictions

In [ ]:
terms = model.named_steps["tfidf"].get_feature_names_out()
weights = model.named_steps["classifier"].coef_.ravel()
strong_positive = np.argsort(weights)[-18:][::-1]
strong_negative = np.argsort(weights)[:18]
important_words = pd.concat([
    pd.DataFrame({"term": terms[strong_positive], "weight": weights[strong_positive], "sentiment": "positive"}),
    pd.DataFrame({"term": terms[strong_negative], "weight": weights[strong_negative], "sentiment": "negative"}),
], ignore_index=True)
display(important_words)

examples = [
    "The story was moving and the performances were excellent. I would watch it again.",
    "The plot was boring and the acting was terrible. I regretted watching it."
]
for review, label in zip(examples, model.predict(examples)):
    print("Positive" if label == 1 else "Negative", "—", review)

## 7. Export artifacts for VS Code / Streamlit

Download the generated ZIP and extract its contents **beside `app.py`**. The complete fitted TF-IDF + classifier pipeline is saved together.

In [ ]:
from zipfile import ZipFile
out = Path("/content/cinescope_sentiment_assets")
out.mkdir(exist_ok=True)

joblib.dump(model, out / "sentiment_model.pkl", compress=3)
(out / "sentiment_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
comparison.to_csv(out / "sentiment_comparison.csv", index=False)
pd.DataFrame(cm, index=["Actual negative", "Actual positive"],
             columns=["Predicted negative", "Predicted positive"]).to_csv(out / "sentiment_confusion_matrix.csv")
pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(out / "sentiment_roc.csv", index=False)
important_words.to_csv(out / "sentiment_terms.csv", index=False)

archive = Path("/content/cinescope_sentiment_assets.zip")
with ZipFile(archive, "w") as z:
    for file in out.iterdir():
        z.write(file, arcname=file.name)
print("Saved:", archive)
print("Test accuracy:", f"{metrics['accuracy']:.2%}")
print("Model:", best_name)
from google.colab import files
files.download(str(archive))

### Viva-ready explanation

**Dataset:** The IMDb 50K Review dataset with genuine `positive` and `negative` labels.  
**Inputs:** Original review text. **Output:** Positive or negative sentiment, **not** a numerical movie rating.  
**TF-IDF:** Converts words and two-word phrases into numeric values.  
**Model choice:** Selected using training/validation data, not the final test data.  
**Evaluation:** Accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix on held-out reviews. Identical reviews are kept in the same split to avoid duplicate-text leakage.  
**Data leakage:** Avoided by fitting TF-IDF only within training splits.  
**Limitation:** A new unseen review may be sarcastic, mixed, or unlike training reviews. The model's decision margin is not a calibrated probability.

**Source:** https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews